# CERIC XAS Python API Quick Start

This notebook shows how to use the CERIC XAS static API from Python. It reads one example XDI spectrum, extracts metadata and numeric columns, plots the spectrum, searches the repository, and downloads matching files.

## How to use this notebook

This notebook is included in `website/notebooks/` so it can be downloaded from the website. It is meant to be run together with a local or published copy of the static site.

For local use, start the website first. Open a terminal in the `website/` folder and run:

```bash
python3 -m http.server 8000
```

Then open `http://localhost:8000/` in the browser. The notebook uses the same address as its `BASE_URL`.

The notebook imports `ceric_xas`, the Python package included in this repository. If the import already works, you do not need to install anything. This can happen when the notebook kernel is started from the repository root, because Python can already see the local `ceric_xas/` folder.

If the import fails, open a terminal in the repository root, the folder that contains `pyproject.toml`, and run:

```bash
python3 -m pip install -e .
python3 -m pip install matplotlib
```

`pip install -e .` installs the local repository as an editable Python package. This makes `ceric_xas` importable from the notebook while still using the local source files. `matplotlib` is only needed for the plot.

For a published website, replace `BASE_URL` with the public website root.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from ceric_xas import Client, parse_xdi

BASE_URL = "http://localhost:8000"
client = Client(BASE_URL)

## Inspect the static API

In [ ]:
client.catalog()

In [ ]:
client.filter_options()

## Read one example XDI spectrum

The example below reads `Cu2S_111.xdi`, the same spectrum used by the homepage preview. The XDI file is downloaded once as text, then `parse_xdi()` extracts metadata, column descriptions, and numeric rows from that same text.

The next cell displays the first 26 lines of the file.

In [ ]:
beamline = "lisa-esrf"
file_name = "Cu2S_111.xdi"

xdi_text = client.read_text(beamline, file_name)
xdi = parse_xdi(xdi_text)

print("\n".join(xdi_text.splitlines()[:26]))

## Inspect metadata

The parsed metadata are stored as a Python dictionary, with XDI metadata names as keys and their values as strings.

In [ ]:
xdi.metadata

## Inspect columns

`xdi.columns` stores the data column names from the XDI header. `xdi.rows` stores all numeric columns, so files with a third column or more keep their full data. `xdi.points` is a convenience property for the first two numeric columns as `(energy, mutrans)` points.

In [ ]:
[(column.index, column.name) for column in xdi.columns]

## Plot the spectrum

The plot uses `xdi.points`, which matches the first two numeric XDI data columns shown in the website cards.

In [ ]:
energy, mutrans = zip(*xdi.points)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(energy, mutrans, color="#3a6f9f", linewidth=1.5)
ax.set_xlabel("E (eV)")
ax.set_ylabel(r"$\mu$ (E)")
ax.grid(alpha=0.15)
ax.set_title(file_name)
fig.tight_layout()

## Search spectra

These are the same fields used by the homepage search. Use `"any"` or `None` to leave a filter open.

In [ ]:
matches = client.search(
    element_symbol="Cu",
    element_edge="K",
    mono_name="111",
    sample_temperature="300 K",
    beamline="LISA@ESRF",
)

print(f"{len(matches)} spectra found")
matches[:5]

## Download matching files

In [ ]:
downloaded = client.download_search(
    element_symbol="Cu",
    element_edge="K",
    mono_name="111",
    sample_temperature="300 K",
    beamline="LISA@ESRF",
    out_dir=Path("downloads") / "lisa-esrf" / "Cu",
)

downloaded

## Reuse with another website

If the website structure stays the same, only change `BASE_URL`. The API expects:

- `catalog.json`, which lists beamlines and manifest paths;
- `pages/{beamline}/manifest.json`, which maps element symbols to XDI files;
- `pages/{beamline}/files/{file}`, which serves the XDI files.

If you add new search fields, update `ceric_xas/client.py` so the `Spectrum` dataclass, metadata parser, `filter_options()`, and `search()` expose the new field consistently.